# Build and Train a Tiny GPT

## Important Information

- Email: [joanna_bieri@redlands.edu](mailto:joanna_bieri@redlands.edu)
- Office Hours take place in Duke 209, [Office Hours Schedule](https://joannabieri.com/schedule.html)
- [Class Website](https://joannabieri.com/machine_learning.html)
- [Syllabus](https://joannabieri.com/machinelearning/IntroMachineLearning.pdf)

:::{.callout-important icon=false}
## How to use these notes

Two kinds of box show up in these notes.

**Blue Q boxes** are questions for you to answer **by hand, in a notebook, with a pen.** Not because I am old fashioned. Writing something down by hand is slow, and slow is the point: it is very hard to write an explanation you do not actually understand. You are welcome to use AI in this class for the mechanics of code, but these boxes are the part where you do the thinking yourself. Bring your written notes to class, I will ask to see them.

**Green You Try boxes** are optional code for you to work through. Nothing is collected and nothing is graded. They are there because you will learn more from changing a number and rerunning than from watching me do it.

**Every code cell begins with a tag** that tells you what to do with it.

- `# RUN THIS.` Setup, loading data, a plot. Copy it, run it, move on. You do not need to be able to write it from memory.
- `# LEARN TO WRITE THIS.` The pattern of the day. The homework will ask you for it, and so will the exam. Type it out yourself at least once rather than pasting it.
- `# DEMO ONLY.` Code that exists to show one idea. You do not need to be able to write it yourself.
- `# UNDERSTAND THESE STEPS.` Code that shows how something works inside, one small step at a time. Read it and make sure you understand what each line does. You do not need to memorize it or be able to write it from scratch.
:::

**Reading:** Geron chapter 15, **Building an English-to-Spanish Transformer**.

When we built the transformer block, we used it to read our made up movie reviews. Remember how that worked: every word went into the block as 16 numbers and came out as 16 numbers. Then we took the average over all of the words, and one `nn.Linear` turned that average into 2 scores, negative or positive. So that transformer read a whole review and gave us back ONE answer.

Let's back up for a minute and remember how the words got into that model, because we are going to do the same thing today.

1. **Every word got an id.** We made a list of all of the words in the reviews and gave each one a number, eg. "the" might be 3 and "great" might be 17. The id is just a label, like a row number. 17 is not "bigger" or "better" than 3.
2. **Every id got turned into 16 numbers.** The id went into an `nn.Embedding` layer. You can think of `nn.Embedding` as a table with one row for every word and 16 numbers in each row. The id tells PyTorch which row to grab. Those 16 numbers start out random, and the network learns them during training, so words that get used in similar ways end up with similar numbers.
3. **The 16 numbers went into the transformer block.** The block never sees the id itself, only the row of numbers.

So it goes: word → id → row of the embedding table → transformer block. Keep this picture in your head, because today we use the exact same steps, just with characters instead of words.

Today we turn this around and build a transformer that **writes** text! **GPT** stands for **Generative Pre-trained Transformer**.

- **Generative** because it makes new text.
- **Pre-trained** because it is trained on a big pile of text before anyone uses it for a specific job (more on this soon).
- **Transformer** because it is made out of the transformer blocks we already built.

Here is the surprising part. Inside, a GPT only ever does one job: it looks at some text and guesses the **next** piece. That's it! So how does it write a whole paragraph? It guesses one piece, adds it to the end of the text, and then guesses again using the longer text. Over and over. ChatGPT works the same way, it is just a LOT bigger and was trained on a LOT more text.

The good news is that we already have almost everything we need. A GPT is a stack of the transformer blocks we built, with three changes:

1. **We ask a different question.** For the reviews we wanted one answer for the whole review. Now we want a guess for the next character at **every** spot in the text, so one piece of text gives the model lots of practice questions at once.
2. **We add a mask.** When the model is guessing what comes after spot 5, it should not be allowed to look at spot 6, because spot 6 IS the answer. The mask hides it from the model. We will see why this matters a lot.
3. **We drop the average over the words.** We need a guess at every spot, so the `.mean(dim=1)` from the review classifier goes away.

Here is our plan for today:

1. Load some Shakespeare and give every character an id.
2. Build the training data: the targets are the text moved over by one character.
3. Make the mask, so no character can look ahead.
4. Build a tiny GPT: an embedding turns every id into 64 numbers, two transformer blocks work on those numbers, and a last layer scores every possible next character. Then train it.
5. Have it write some Shakespeare!
6. See what goes wrong when we forget the mask.

# The Data - Shakespeare, One Character at a Time

To teach a model to write we need a lot of text, and it helps if the text has a style we can recognize. So we are going to use about 1 MB of Shakespeare plays, all put together in one file. This is the same file Geron uses in chapter 14 to train a character RNN (it was put together by Andrej Karpathy, and Shakespeare is out of copyright, so we are allowed to use it). It is in the `data` folder that comes with these notes.

For the reviews, the pieces were **words**. Today the pieces are **characters**: every letter, space, comma and newline is one piece. Why switch? Shakespeare uses about 12,600 different words, and about 5,700 of them show up only once. A model that works with words would need an id, and a row of the embedding table, for every one of them. And it could only ever write words it had already seen. With characters there are only a few dozen different pieces, so we can print them all out and see exactly what the model is choosing from, and the model can spell any word, even ones it makes up!

Everything else works the same way as the reviews: every character gets an id, and the id picks out a row of an embedding table.

Real GPTs use pieces in between words and characters, called **tokens** (often a whole word or part of a word). That is a topic all on its own that we can look at later.

In [ ]:
# RUN THIS. Imports and the text.
import math                                     # for math.log
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

text = open("data/shakespeare.txt").read()      # the whole file as one long string
print("characters in the file:", len(text))
print(text[:250])                               # the first 250 characters, so we can see what it looks like

Look at the start of the file. It is laid out like a play: the name of the person talking, a colon, and then their lines. Keep this layout in mind, because later we will check whether our model learns it!

Computers work with numbers, not letters. So just like every word got an id in the reviews, every different character gets an id here. Here is how we do it:

1. `set(text)` keeps exactly one copy of each character in the whole file.
2. `sorted` puts them in order. A set does not keep things in any order, so without `sorted` the ids could come out different each time we run this.
3. We number the characters 0, 1, 2, ... and save that in a dictionary.

Then we write two small functions. `encode` turns text into a list of ids, and `decode` turns a list of ids back into text. We need both: `encode` for the text going into the model, and `decode` so we can read what comes out.

In [ ]:
# RUN THIS. Give every character an id, and two functions to go back and forth.
chars = sorted(set(text))          # every different character, in order
vocab_size = len(chars)
print("number of different characters:", vocab_size)
print(chars)

char_to_id = {}                    # a dictionary: character -> id
for i in range(vocab_size):
    char_to_id[chars[i]] = i

def encode(some_text):             # text -> list of ids
    ids = []
    for c in some_text:
        ids.append(char_to_id[c])
    return ids

def decode(ids):                   # list of ids -> text
    pieces = []
    for i in ids:
        pieces.append(chars[i])
    return "".join(pieces)

print(encode("First"))
print(decode(encode("First")))    # encode and then decode should give back the same text

Notice the first character in the list is `'\n'`, the newline, and the second one is a space. Those count as characters too! The model has to learn where to put the spaces and when to start a new line, just like it has to learn the letters.

Also notice that `encode` and then `decode` gave us back "First", so our two functions undo each other.

BEWARE: these ids are NOT what the model learns from. Just like with the reviews, each id will go into an `nn.Embedding` layer that grabs that character's row of learned numbers. "F" is id 18, and id 18 just means "use row 18 of the embedding table". We set this up when we build the model below.

---

# The Question - What Comes Next?

So what exactly are we going to ask the model? For the reviews, every review came with a label that a person had to decide (positive or negative). Here nobody has to decide anything, because the **text is its own label**.

Take any piece of text. The input is the text, and the target is the same text moved over by one character. So at every spot, the right answer is just the character that comes next:

![The targets are the input moved over by one spot. Every spot is its own question.](images/01-next-character.png)

This means one piece of 5 characters is really 5 questions at once: after "F" comes "i", after "Fi" comes "r", after "Fir" comes "s", and so on. Notice that each question gets to use everything that came before it, so the later questions have more to work with.

This is a really big deal! Nobody has to label anything to train a GPT. Any text at all can be training data: books, websites, code, whatever you have. That is a big part of why these models could get so big, there is a LOT of text out there.

Now we need to cut our one long text into pieces the model can work with. We cut it into pieces of 65 characters:

- The first 64 characters of each piece are the input `X`.
- The last 64 characters (the same piece moved over by one) are the target `y`.

Why 65? To ask 64 questions we need 64 characters going in, plus one more character at the end to be the answer to the last question.

The **64** is called the **context window**. It is how many characters the model can see at once. Our model can never look back further than 64 characters, so it can't remember who was talking a page ago. Real GPTs have much bigger context windows, and we will compare them at the end of these notes.

In [ ]:
# If you want to look at our data uncomment the line below
# BEWARE - it will print out everything!
# text

In [ ]:
# DIVIDE UP THE DATA. 
# Cut the text into pieces of 65 characters: 64 in, 64 targets.
# It is nice to know how to write code like this!

context_length = 64                                  # how many characters the model sees at once

ids = torch.tensor(encode(text))                      # the whole text as ids
n_pieces = len(ids) // (context_length + 1)           # how many pieces of 65 fit
pieces = ids[:n_pieces * (context_length + 1)]        # drop the few left over at the end
pieces = pieces.reshape(n_pieces, context_length + 1) # one piece per row

X = pieces[:, :-1]       # the first 64 characters of every piece
y = pieces[:, 1:]        # the last 64: the same piece moved over by one
print("X:", X.shape)
print("y:", y.shape)

# Look at the first 20 characters for the first X,y pair
print("input: ", repr(decode(X[0, :20].tolist())))
print("target:", repr(decode(y[0, :20].tolist())))

Let's look at the first 20 characters of the first piece. The target is the input moved over by one: "First Citizen:\nBefor" goes in and "irst Citizen:\nBefore" comes out. (`repr` prints the newline as `\n` so we can see it. Without `repr` the print would just start a new line and we would miss it.)

Next we need a train and validation split. Usually we split at random, but this time we keep the **last 10%** of the pieces for validation. Why not random? The text is one long story, so pieces that sit next to each other share a lot: the same people talking, the same names, the same scene. If we picked at random, almost every validation piece would have its neighbors in the training data, so the model would already have seen most of what it is being tested on. Taking the end of the file means the validation pieces come from a part of the plays the model has never seen, so the validation loss is a more honest test.

NOTE: This is a pro move that you should try to remember! Any time your data has an order (eg. time series data), think about whether a random split lets information leak from training into validation.

In [ ]:
# LEARN TO WRITE THIS. 
# We dont use train-test-split
# Instead, the last 10% of the pieces are the validation set.
n_train = int(0.9 * n_pieces)
X_train, y_train = X[:n_train], y[:n_train]
X_valid, y_valid = X[n_train:], y[n_train:]
print("training pieces:  ", X_train.shape[0])
print("validation pieces:", X_valid.shape[0])

:::{.callout-note icon=false}
## Q1. Write this one out by hand

**a.** Write the input and the target for the piece "to be or" (8 characters in, 8 targets). How many questions does this one piece ask the model?

**b.** For the reviews, a person had to give every review a label. Why does a GPT not need anyone to label its training data?
:::

---

# No Peeking - The Mask

Now we have a problem. When we trained on the reviews, every word could look at every other word, and that was great. That is the whole reason attention works so well! But now every spot has to guess the **next** character.

Think about spot 0, the "F" in "First". Its job is to guess that "i" comes next. If it is allowed to look at spot 1, it can see the "i"! Then guessing is easy, just copy it. And these models will cheat any chance they get if it improves their loss function! The model would get a really low loss and learn nothing at all about writing.

So we need to hide the characters that come later. The rule is:

- each character can look at itself and every character **before** it,
- and it can not look at anything **after** it.

This is called a **causal mask** (cause comes before effect, so nothing can be changed by something that comes later). Here it is for "First". Each row is one character, and the row shows which characters it is allowed to look at:

![The causal mask. Row "r" can look at "F", "i" and "r", but not at "s" or "t".](images/02-causal-mask.png)

Read the "r" row. "r" can look at "F", "i" and "r", the characters up to and including itself. It can not look at "s" or "t", because those come later in the word.

In PyTorch the mask is a table of True and False, with **True where looking is NOT allowed**. Look at the picture again: the hidden spots are all of the spots above the diagonal. PyTorch has a command that keeps only the upper triangle of a table, `torch.triu` (tri for triangle, u for upper), so making the mask only takes one line.

In [ ]:
# LEARN TO WRITE THIS. The causal mask for 5 characters.
mask = torch.triu(
    torch.ones(5, 5, dtype=torch.bool),     # a 5 by 5 table of True
    diagonal=1)                             # keep only what is ABOVE the diagonal, the rest becomes False
print(mask)

Look at the printout. The diagonal and everything below it is False (allowed), and everything above the diagonal is True (hidden). The `diagonal=1` is what keeps the diagonal itself False, so each character can still look at itself.

BEWARE: in this mask **True means hidden**. This feels backwards to a lot of people and it is easy to get wrong. If you flip it, you hide the past and show the future, the exact opposite of what we want!

Now let's see what the mask actually does. First we will do a small example by hand, and then we will let PyTorch do it.

## A Small Example by Hand

Let's use just the first three characters, "Fir", and follow the same steps the model takes:

1. **Characters to ids.** `encode("Fir")` gives the ids 18, 47 and 56. (Check this against the printout above, "First" was [18, 47, 56, 57, 58].)
2. **Ids to numbers.** Each id picks out one row of the embedding table. To keep the arithmetic easy, I made up an embedding with only 2 numbers per character: "F" gets [1, 0], "i" gets [0, 1], and "r" gets [1, 1]. In the real model `nn.Embedding` has 64 numbers per character, and nobody picks them, the network learns them. I made these up so we can follow every step.
3. **Compare every pair.** Attention compares characters with a dot product. In the real model each character makes a query and a key first, but to keep this small we just compare the vectors themselves. For example "r" with "i" is $1 \cdot 0 + 1 \cdot 1 = 1$, and "r" with itself is $1 \cdot 1 + 1 \cdot 1 = 2$.
4. **Hide the future.** Every spot above the diagonal gets $-\infty$. So "F" can't see "i" or "r", and "i" can't see "r".
5. **Softmax each row.** $e^{-\infty} = 0$, so the hidden spots get a weight of exactly 0.

![The mask by hand on "Fir": ids, made up embedding rows, the scores with the future hidden, and the weights after softmax.](images/05-mask-by-hand.png)

Read the weights one row at a time:

- "F" can only look at itself, so its row is [1, 0, 0].
- "i" can look at "F" and itself. Its scores are [0, 1], and softmax turns them into [0.27, 0.73].
- "r" can look at everything up to itself. Its scores are [1, 1, 2], so it puts the most weight on itself (0.58) and splits the rest between "F" and "i" (0.21 each).

What if we forgot the mask? Then the "F" row would have the scores [1, 0, 1], and softmax would give [0.42, 0.16, 0.42]. "F" would put almost half of its weight on "r", a character that comes AFTER it. That is exactly the cheating (peeking) we are trying to stop!

## Now in PyTorch

The code below does the same thing for all 5 characters of "First", with two differences. The embedding has 8 numbers per character instead of 2 (random for now, because the layer is brand new), and `nn.MultiheadAttention` makes its own queries, keys and values from those numbers before it compares them.

1. We send "First" through a brand new `nn.MultiheadAttention` with the mask. We hand the mask to attention with `attn_mask=`.
2. We look at the attention weights.


In [ ]:
# UNDERSTAND THESE STEPS. Attention with the mask on "First".
torch.manual_seed(42)
embed = nn.Embedding(vocab_size, 8)                    # 8 numbers per character, just for this example
vectors = embed(torch.tensor([encode("First")]))      # 1 piece, 5 characters, 8 numbers
attention = nn.MultiheadAttention(8, 1, batch_first=True)    # 1 head, so we get one table

output, weights = attention(vectors, vectors, vectors,
                            attn_mask=mask)            # True spots get a weight of 0
print(torch.round(weights[0].detach(), decimals=2))

Everything above the diagonal is 0, so the mask worked! Read the table one row at a time, the same way we did by hand:

- The "F" row has a single 1 in it. "F" is the first character, so the only thing it can look at is itself, and all of its weight goes there.
- The "i" row splits its weight between "F" and "i" (0.47 and 0.53).
- Each row after that gets to spread its weight over one more character.
- Every row still adds up to 1.

This layer is brand new, so the weights themselves are not useful yet. They only start to mean something after training. But because of the mask, no matter what the model learns, no row can ever put weight on a character that comes after it.

How does PyTorch do this inside? Remember that attention makes a table of scores and sends each row through a softmax. Before the softmax, PyTorch puts $-\infty$ in every hidden spot of the score table. The softmax uses $e^{\text{score}}$, and $e^{-\infty} = 0$, so those spots get a weight of exactly 0. The softmax then spreads all of the weight over the spots that are left, which is why every row still adds up to 1.

:::{.callout-note icon=false}
## Q2. Write this one out by hand

**a.** Why would a model trained without the mask get a very low training loss but still not be able to write anything?

**b.** In the mask for "First", which characters can "s" look at? How many True values are in a mask for 64 characters? (Hint: count the spots above the diagonal of a 64 by 64 table.)
:::

---

# The Tiny GPT

Now we have all of the pieces, so let's put it together!

We start with our `TransformerBlock` from when we built the transformer block. The only change is one line: `forward` now takes a `mask` and hands it to attention. We allow it to be None, so we can turn the mask on and off and compare the results. Everything else, the skip, the layer norm and the feed-forward part, is exactly the same as before. We do not have to build anything new to make a model that writes, we just have to tell attention what it is not allowed to look at.

In [ ]:
# LEARN TO WRITE THIS. Our transformer block, with ONE change: it takes a mask.
class TransformerBlock(nn.Module):

    def __init__(self, n_numbers=16, n_heads=2):
        super().__init__()
        self.attention = nn.MultiheadAttention(n_numbers, n_heads, batch_first=True)   # the heads, side by side
        self.norm1 = nn.LayerNorm(n_numbers)              # layer norm after attention
        self.feed_forward = nn.Sequential(                # the small MLP, one character at a time
            nn.Linear(n_numbers, 2 * n_numbers),
            nn.ReLU(),
            nn.Linear(2 * n_numbers, n_numbers))
        self.norm2 = nn.LayerNorm(n_numbers)              # layer norm after the feed-forward

    def forward(self, X, mask=None):                                        # NEW: the mask
        attention_out, weights = self.attention(X, X, X, attn_mask=mask)    # NEW: attn_mask=mask
        Z = self.norm1(X + attention_out)                 # the skip, then layer norm
        out = self.norm2(Z + self.feed_forward(Z))        # feed-forward, the skip, then layer norm
        return out

Below we will write the code, but first let's walk through the whole model, one piece at a time. We will read it from the top to the bottom: the characters go in, get their numbers, go through two blocks, and come out as a guess for the next character.

We will follow the very first piece of the play, which starts with "First Citizen:" and ends with "speak.", two new lines, and "Al". The model takes all 64 characters at once, so the pictures show the first few and the last few with "..." in between.

Click on each piece to open it. <a href="images/06-tiny-gpt-detail.png" target="_blank">Click here to see the whole picture at once.</a>

<details style="background-color: #eef6ff; border: 2px solid #8ab8e6; border-radius: 8px; padding: 6px 14px; margin-bottom: 10px;">
<summary><b>Input</b></summary>

<img src="images/06a-input.png" alt="The first piece of the play as letter boxes with their ids underneath." width="100%">

**The characters go in as ids.** "F" is 18, "i" is 47, and so on. Remember, the ids are just labels, they are the row numbers we will use in the next step.

</details>

<details style="background-color: #fdeee8; border: 2px solid #e8a48a; border-radius: 8px; padding: 6px 14px; margin-bottom: 10px;">
<summary><b>Embedding</b></summary>

<img src="images/06b-embedding.png" alt="Each id goes down into a column of 64 numbers." width="100%">

**Each id gets its numbers.** The id picks out its row of the character table, so every character turns into a list of 64 numbers (the colored column in the picture). We also add a row from the position table, so the model knows WHERE each character is. Now "F" is 64 numbers that say "I am an F, and I am the first character".

</details>

<details style="background-color: #fff3e3; border: 2px solid #f0b46a; border-radius: 8px; padding: 6px 14px; margin-bottom: 10px;">
<summary><b>Transformer Block 1</b></summary>

<img src="images/06c-block1.png" alt="Transformer block 1: four heads with the mask, skip, layer norm, feed-forward, skip, layer norm." width="100%">

**Block 1 lets each character look back.** Follow the picture from the top:

1. **The 4 heads.** This is the attention we built. Each character looks at itself and the characters before it (the mask hides everything after it) and gathers up what is useful for guessing the next character. With 4 heads, each character can look for 4 different things at the same time. Each head works with 16 numbers. We don't decide what the heads look for, training does. For example, one head might end up paying attention to the character right before, and another to whether we are in the middle of a name.
2. **Side by side.** The 4 heads each give back 16 numbers, and we put them side by side to get 64 numbers again.
3. **Skip and layer norm.** The blue skip arrow adds what came into the block to what attention made, so each character remembers what it was. Then layer norm keeps the numbers a good size.
4. **Feed-forward.** A small MLP works on each character on its own, going from 64 numbers up to 128 and back down to 64.
5. **Skip and layer norm again**, around the feed-forward part.

Each character still has 64 numbers, but now those numbers also carry information about the characters before it.

</details>

<details style="background-color: #fff3e3; border: 2px solid #f0b46a; border-radius: 8px; padding: 6px 14px; margin-bottom: 10px;">
<summary><b>Transformer Block 2</b></summary>

<img src="images/06d-block2.png" alt="Transformer block 2: the same steps as block 1." width="100%">

**Block 2 does the same steps again**, starting from what block 1 made. It has its own weights, so its heads can look for different things than the heads in block 1. And because it starts from what block 1 already gathered, block 2 can combine things that block 1 found.

</details>

<details style="background-color: #eaf8e6; border: 2px solid #7cc47a; border-radius: 8px; padding: 6px 14px; margin-bottom: 10px;">
<summary><b>Output Layer</b></summary>

<img src="images/06e-output.png" alt="Out of the blocks: 64 numbers per character, then 65 scores at every spot. The last spot is boxed in red." width="100%">

**The numbers become scores.** Out of the blocks, every character still has 64 numbers. Then at every spot, one `nn.Linear` turns those 64 numbers into 65 scores, one for every character in our list. A big score means "I think this character comes next".

When we train, every spot counts: each spot has a real next character, and `nn.CrossEntropyLoss` does a softmax and looks at how much probability the model gave to that real next character.

</details>

<details style="background-color: #fffbe0; border: 2px solid #e6c84a; border-radius: 8px; padding: 6px 14px; margin-bottom: 10px;">
<summary><b>Softmax and Pick the Next Character</b></summary>

<img src="images/06f-pick.png" alt="At the last spot the 65 scores become probabilities, we pick l at random, and Al becomes All." width="100%">

**The scores become a guess.** Softmax turns the 65 scores into 65 probabilities that add up to 1, just like it did for the 2 scores in the review classifier. The softmax is not inside our model code. When we train, `nn.CrossEntropyLoss` does it for us. When we write, the `generate` function does it.

When we write, we only care about the LAST spot, the red box, because its scores are the guess for the character that comes next. Here the last characters are "Al", and the model thinks "l" is very likely, so we will probably get "All" (in the play, "All:" is the whole crowd talking at once). We pick the next character at random using the probabilities, add it to the end, and go again (more on this in **Let It Write!**).

</details>

Now here is the code. Try to find each piece of the picture in it.


In [ ]:
# LEARN TO WRITE THIS. The tiny GPT.
class TinyGPT(nn.Module):

    def __init__(self, n_numbers=64, n_heads=4, n_blocks=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, n_numbers)               # numbers for each character
        self.position_embed = nn.Embedding(context_length, n_numbers)  # numbers for each spot
        self.blocks = nn.ModuleList()                                  # a list PyTorch can see inside
        for i in range(n_blocks):
            self.blocks.append(TransformerBlock(n_numbers, n_heads))
        self.out = nn.Linear(n_numbers, vocab_size)                    # a score for every character

    def forward(self, X):
        n_spots = X.shape[1]                              # how many characters in each piece
        spots = torch.arange(n_spots)                     # 0, 1, 2, ... one number for each spot
        mask = torch.triu(torch.ones(n_spots, n_spots, dtype=torch.bool), diagonal=1)   # the causal mask, the right size
        Z = self.embed(X) + self.position_embed(spots)    # (pieces, spots, 64)
        for block in self.blocks:
            Z = block(Z, mask)                            # (pieces, spots, 64)
        scores = self.out(Z)                              # (pieces, spots, 65)
        return scores

Let's match the code to the picture.

**In `__init__` we make the layers.** This is where all of the numbers the model learns are kept.

- `self.embed` is the character table: 65 rows one for each character (letters, spaces, punctuation and new lines), 64 numbers in each row the length of the letter embedding vector.
- `self.position_embed` is the position table: 64 rows one for each spot in the context, 64 numbers in each row the length of the positional embedding vector.
- `self.blocks` holds our transformer blocks in a PyTorch list. The `for` loop makes `n_blocks` of them. We start with two transformer blocks but could experiment with more.
- `self.out` is the last `nn.Linear`, which turns 64 numbers (vectors coming out of the last transformer) into 65 scores (a score for each character - when we softmax this it becomes a probability that each character is the next character).

**In `forward` we send the text through the layers, from the top of the picture to the bottom.**

1. Make the list of spots 0, 1, 2, ..., 63, and the mask for that many characters.
2. Look up the embedding numbers for each character and the positional embedding numbers for each spot, and add them together.
3. Send the numbers through each block, one block at a time, with the mask.
4. Turn the 64 numbers at every spot into 65 scores, and return them.

## What Is New Compared to the Review Classifier?

1. **64 numbers per character and 4 heads** instead of 16 numbers and 2 heads. Writing is a much harder job than spotting a feeling word in a review, so we give the model more room to store what it learns. Each head works with $64 / 4 = 16$ numbers.
2. **Two blocks, one on top of the other.** The output of block 1 is the input to block 2. Why stack them? In each block, every character gathers information from the characters before it. With two blocks, the second block gets to work with what the first block already put together. Real GPTs stack a lot more of them: GPT-2 small has 12 blocks and GPT-3 has 96 (see the table at the end of these notes).
3. **A list of blocks, `nn.ModuleList`.** Why do we need a list at all? We could write `self.block1 = TransformerBlock(...)` and `self.block2 = TransformerBlock(...)` by hand, and that would work fine for 2 blocks. But we want to be able to try 1 block, or 4, or 12, just by changing `n_blocks`. A list and a `for` loop let us make as many blocks as we ask for, and then send the numbers through all of them.
4. **The output layer goes from 64 numbers to `vocab_size` = 65 scores**, one score for every character that could come next. To get from scores to the next character we use softmax and then pick one at random, using those probabilities.
5. **The mask gets made inside `forward`.** The model makes a new mask every time, the right size for however many characters we send in. This matters when we write text, because then we sometimes send in fewer than 64 characters.

In [ ]:
# UNDERSTAND THESE STEPS. 
# Here we send 3 pieces through a brand new model and look at the shapes.
torch.manual_seed(42)
model = TinyGPT()
scores = model(X_train[:3])
print("going in: ", X_train[:3].shape)
print("coming out:", scores.shape)

n_params = 0                      # add up the size of every weight tensor
for p in model.parameters():
    n_params = n_params + p.numel()
print("numbers to learn:", n_params)

Let's read the shapes. 3 pieces of 64 characters go in: (3, 64). What comes out is (3, 64, 65): for each of the 3 pieces, at each of the 64 spots, we get 65 scores, one for every character that could come next. That is a guess at every spot, just like we wanted.

Before we train, how good is a brand new model? It helps to have a number to beat. Think about a model that knows nothing at all. It would give all 65 characters the same chance, so the right character gets a probability of $1/65$. The cross entropy loss is minus the log of the probability given to the right answer, so

$$-\ln\left(\frac{1}{65}\right) = \ln(65) \approx 4.174$$

If our trained model can't get below that, it has not learned anything.

In [ ]:
# RUN THIS. 
#  The loss of a model that guesses every character equally, 
# and the loss of our brand new model before we have done any training.
print("loss from guessing evenly:", round(math.log(vocab_size), 3))

loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(scores.reshape(-1, vocab_size), y_train[:3].reshape(-1))
print("loss of the new model:", round(loss.item(), 3))

The brand new model is about as bad as guessing evenly, which is exactly what we expect. It has not seen any Shakespeare yet!

NOTE: Notice the `.reshape` in the loss. **This is new**. Here is why we need it... `nn.CrossEntropyLoss` wants:

- one row of scores for each question, and
- one target for each question.

Our scores have the shape (pieces, spots, 65). We sent in 3 pieces of text, each piece has 64 spots (context), and the model outputs 65 scores. Every spot is asking a question, what comes next?, so really we have 3*64=192 questions that we tried to answer. When we calculate the loss we want to see how we did on each and every one of these questions so we need to reshape the data. To do this we flatten the pieces and the spots together into one long list of questions:

- the scores (3, 64, 65) become (192, 65), because $3 \times 64 = 192$ questions,
- the targets (3, 64) become 192 targets.

The `-1` tells PyTorch "work this number out for me". It knows there are 65 columns, so it figures out that there must be 192 rows. It is better to do this automatically so we don't have to recalculate if we change the context size.

:::{.callout-note icon=false}
## Q3. Write this one out by hand

**a.** The review classifier took the mean over the words before its last `nn.Linear`. Why does the GPT not take the mean?

**b.** A character model on text with only 27 different characters (a to z and a space). What is the loss of a model that guesses every character evenly? What would a loss of 0 mean?
:::

---

# Train It!

Now we can teach our tiny GPT to write. The training loop is the same one we have been using since we learned PyTorch, with only two changes:

1. The `.reshape` in the loss, so every spot counts as its own question.
2. We keep track of the **loss** after every epoch instead of the accuracy.

Why not accuracy? Think about what comes after "the ". Lots of characters are reasonable! "the king", "the house", "the end"... If the model guesses "k" and the text has "h", accuracy counts that as completely wrong, even though "k" was a good guess. The loss works better here because it looks at how much probability the model gave to the character that really came next. So a model that says "h is pretty likely" gets credit for it, even if "h" was not its top pick.

One epoch is one pass through all of the training pieces. **On a laptop each epoch takes a few seconds to a minute, depending on how new your laptop is, so be patient!**

In [ ]:
# RUN THIS. The training loop. It returns the training and validation loss after every epoch.
def train_gpt(model, X_train, y_train, X_valid, y_valid, n_epochs):
    torch.manual_seed(42)                         # so the batches come in the same order every time
    train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

    train_losses = []
    valid_losses = []
    for epoch in range(n_epochs):
        model.train()                             # training mode
        total = 0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()                 # clear the old gradients
            scores = model(X_batch)               # (32, 64, 65)
            loss = loss_fn(scores.reshape(-1, vocab_size),    # every spot is its own question
                           y_batch.reshape(-1))
            loss.backward()                       # backward pass
            optimizer.step()                      # one step of Adam
            total = total + loss.item()
        train_losses.append(total / len(train_loader))   # the average over the batches

        model.eval()                              # prediction mode
        with torch.no_grad():
            scores = model(X_valid)
            valid_loss = loss_fn(scores.reshape(-1, vocab_size), y_valid.reshape(-1)).item()
        valid_losses.append(valid_loss)
        print("epoch", epoch + 1, "  train loss:", round(train_losses[-1], 3), "  valid loss:", round(valid_loss, 3))

    return train_losses, valid_losses

In [ ]:
# LEARN TO WRITE THIS. Train the tiny GPT for 5 epochs.
torch.manual_seed(42)
model = TinyGPT(n_numbers=64, n_heads=4, n_blocks=2)
train_losses, valid_losses = train_gpt(model, X_train, y_train, X_valid, y_valid, n_epochs=5)

In [ ]:
# RUN THIS. The learning curves, with the loss from guessing evenly for comparison.
epochs = range(1, 6)
plt.figure(figsize=(6, 4))
plt.plot(epochs, train_losses, marker="o", label="training")
plt.plot(epochs, valid_losses, marker="o", label="validation")
plt.axhline(math.log(vocab_size), color="grey", linestyle="--", label="guessing evenly")
plt.xlabel("epoch")
plt.xticks(epochs)                   # only whole epochs on the x axis
plt.ylabel("loss")
plt.ylim(0, 4.5)
plt.legend()
plt.grid()
plt.show()

Look at that! The validation loss goes from 2.087 after the first epoch down to 1.827 after 5 epochs. That is way below the 4.174 we get from guessing evenly.

But what does a loss of 1.827 actually mean? Let's undo the log. The loss is minus the log of the probability given to the right character, so

$$e^{-1.827} \approx 0.16$$

So, roughly, the model now gives the character that really comes next about a 16% chance. Guessing evenly gives it $1/65 \approx 1.5\%$. That is more than 10 times better! 16% might sound low, but remember that there are lots of places in text where more than one next character is a good guess, so no model could get close to 100% everywhere. (This is a rough way to read the number, because the loss is an average of logs over lots of characters. But it gives you a good feel for it.)

Are we overfitting? Both curves are still going down at epoch 5, and the validation loss has not started going back up, so no. More epochs would probably still help. You can try this in the You Try box at the end.

NOTE: The training loss after epoch 1 is higher than the validation loss. That looks strange! It happens because the training loss is the average over the whole epoch, including the very first batches when the model knew nothing. The validation loss is measured at the end of the epoch, after all of that learning.

# Let It Write!

Now the fun part! We trained the model to guess the next character. To write text we use that one skill over and over, the same way ChatGPT writes its answers:

1. Start with some text, eg. "ROMEO:".
2. Send it through the model and take the scores at the **last** spot. Those are the scores for the character that comes next. (The model also makes guesses at all of the other spots, but we already know what comes next there, so we ignore them.)
3. Turn the scores into probabilities with softmax and **pick a character at random** using those probabilities.
4. Add that character to the end of the text and go back to step 2.

![Writing text one character at a time. Every new character goes on the end, and we ask again.](images/04-generate-loop.png)

Why pick at random instead of always taking the most likely character? If we always take the most likely one, we get the exact same text every time, and it tends to repeat the same few words over and over. Picking at random, using the model's probabilities, gives text that sounds much more natural. It also means you get something different every time you run it (unless you set the seed).

`torch.multinomial` does the random pick. If a character has a probability of 0.3, it gets picked about 30% of the time.

One more detail: we only send in the **last 64 characters**. The model only has position numbers for 64 spots, so it can't take in anything longer. Once our text gets longer than 64 characters, we send in the newest 64 and the older characters are left out.

**Here is code for our text generator using our GPT model!**

In [ ]:
# LEARN TO WRITE THIS. Write n_new characters, one at a time.
def generate(model, start, n_new, temperature=1.0):
    ids = encode(start)                                   # the starting text as ids
    model.eval()                                          # prediction mode
    with torch.no_grad():
        for step in range(n_new):
            context = torch.tensor([ids[-context_length:]])   # the last 64 characters, as 1 piece
            scores = model(context)                           # (1, spots, 65)
            last_scores = scores[0, -1]                       # the scores for the NEXT character
            probs = torch.softmax(last_scores / temperature, dim=0)
            next_id = torch.multinomial(probs, num_samples=1).item()   # pick one at random, using probs
            ids.append(next_id)                               # add it to the end and go again
    return decode(ids)

In [ ]:
# LEARN TO WRITE THIS. Our tiny GPT writes some Shakespeare!
torch.manual_seed(0)              # so the random picks are the same every time
print(generate(model, "ROMEO:", n_new=400))

It is not Shakespeare, but look at everything it picked up just from guessing the next character over and over!

- The text is laid out like a play: a name in capitals, a colon, a new line, then the lines.
- It spells real words ("blood", "queen", "light", "love").
- It even uses a real name from the history plays ("Warwick").
- It puts commas and periods in reasonable places.

We never told it any of this. It learned all of it from the text.

It also makes up words ("rechess", "ancosen") and names ("PERS", "DED"), and none of it means anything. Remember that it only sees 64 characters at a time, and it has never been told what any word means.

This model has 79,425 numbers to learn and it trained for 5 epochs on a laptop. Keep that number in mind when we compare it to the real GPTs at the end of these notes.

## Temperature

You might have noticed the `temperature` input in the `generate` function. It is the same "temperature" setting you see in tools like ChatGPT, and now we can see exactly what it does. We divide the scores by the temperature $T$ before the softmax:

$$\text{probabilities} = \text{softmax}\left(\frac{\text{scores}}{T}\right)$$

Let's try it on some made up scores for three characters: [2, 1, 0]. The first character is the model's favorite, the second one is okay, and the third one is unlikely.

| temperature | scores divided by $T$ | probabilities |
|---|---|---|
| 1.0 | [2, 1, 0] | [0.665, 0.245, 0.090] |
| 0.5 | [4, 2, 0] | [0.867, 0.117, 0.016] |
| 2.0 | [1, 0.5, 0] | [0.506, 0.307, 0.186] |

- **Temperature below 1** makes the differences between the scores bigger, so the favorite gets even more of the probability. The text is safer and repeats more.
- **Temperature above 1** squishes the scores closer together, so the unlikely characters get picked more often. The text gets more surprising, and if you go too high it stops making sense.
- **Temperature 1** leaves the scores alone, so we use the model's probabilities just as they are.

Remember when we divided by $\sqrt{d}$ in attention to keep the softmax from getting too sharp? This is the same idea, but now we get to choose the number.

In [ ]:
# DEMO ONLY. The same start at three temperatures.
for temperature in [0.5, 1.0, 1.5]:
    torch.manual_seed(0)
    print("temperature", temperature)
    print(generate(model, "ROMEO:", n_new=150, temperature=temperature))
    print("----------------------------------")

The model and the start are the same in all three, only the temperature changed. Let's compare:

- At **0.5** the words are almost all real and we even get real names (KING HENRY VI, twice in a row). The likely characters get picked almost every time.
- At **1.0** we get the same text as before, because dividing by 1 changes nothing.
- At **1.5** most of the words are made up ("selfeilouce", "seepelinss"). The unlikely characters are getting picked a lot more often.

So which one is best? It depends on what you want! If you want text that is safe and predictable, go low. If you want something more creative, go a little above 1, but not so high that the text stops making sense.

:::{.callout-note icon=false}
## Q4. Write this one out by hand

**a.** Explain to a friend how the model writes 400 characters when it can only guess ONE next character at a time.

**b.** What does a temperature of 0.5 do to the probabilities, compared to a temperature of 1.5? Which one would you use to write something creative?
:::

---

# What If We Forget the Mask?

Earlier we said that without the mask the model could just copy the answer. That is a big claim, so let's check it! Below we make a model that is exactly the same as our tiny GPT, except that its blocks get **no mask**. Every character can see the whole piece, including the character it is supposed to guess. We train it for only 1 epoch and see what happens.

In [ ]:
# DEMO ONLY. A tiny GPT with NO mask. Every character can see the one it is supposed to guess.
class NoMaskGPT(TinyGPT):
    def forward(self, X):
        spots = torch.arange(X.shape[1])
        Z = self.embed(X) + self.position_embed(spots)
        for block in self.blocks:
            Z = block(Z, mask=None)           # no mask: every spot can see every other spot
        return self.out(Z)

torch.manual_seed(42)
cheater = NoMaskGPT()
cheat_train, cheat_valid = train_gpt(cheater, X_train, y_train, X_valid, y_valid, n_epochs=1)

In [ ]:
# DEMO ONLY. Now let it write.
torch.manual_seed(0)
print(generate(cheater, "ROMEO:", n_new=200))

Wow! After only 1 epoch, the model with no mask has a validation loss of 0.045. That is WAY lower than the 1.827 our real tiny GPT got after 5 epochs. If we only looked at the loss, we would think this is an amazing model!

Then we ask it to write and we get "ROMEO:EEEEEEE...". So what happened?

The low loss came from copying. During training, 63 of the 64 spots could see the exact character they were supposed to guess, sitting right next to them. The easiest way to get a low loss was to copy that character, so that is what the model learned to do.

Now think about what happens when we write. We ask for the next character at the LAST spot, and there is nothing after it yet. There is nothing to copy! The model never learned how to guess from what came before, so it writes garbage.

BEWARE: if a loss looks much better than you expected, look for a leak! This is the same kind of problem as letting your test set leak into training. The model is getting scored using information it should not have, and the score looks great until you try to use the model for real.

:::{.callout-note icon=false}
## Q5. Write this one out by hand

**a.** The model with no mask got a much lower validation loss than our real tiny GPT. Why is that low loss not a sign of a good model?

**b.** When the model writes text, there is no next character to peek at. Use that to explain why the no mask model writes garbage.
:::

---

# From Our Tiny GPT to the Real Thing

So how does our little model compare to the GPTs you have heard of? The amazing thing is that they are built the same way:

- embeddings plus positions,
- a stack of transformer blocks with a causal mask,
- and a layer that scores every possible next piece.

The big difference is size. Here are the numbers:

| | our tiny GPT | GPT-2 small (2019) | GPT-3 (2020) |
|---|---|---|---|
| pieces of text | characters (65 of them) | tokens (50,257 of them) | tokens (50,257 of them) |
| context window | 64 | 1,024 | 2,048 |
| numbers per piece | 64 | 768 | 12,288 |
| heads | 4 | 12 | 96 |
| blocks | 2 | 12 | 96 |
| numbers to learn | 79,425 | about 124 million | about 175 billion |

(GPT-2 from Radford et al. 2019, "Language Models are Unsupervised Multitask Learners". GPT-3 from Brown et al. 2020, "Language Models are Few-Shot Learners". OpenAI stopped publishing these numbers for its newer models.)

Look at the last row. GPT-2 small has about 1,500 times as many numbers to learn as our tiny GPT, and GPT-3 has more than 2 million times as many!

Now notice what is NOT in the training. The loss only rewards guessing the next piece the way the training text would continue. Nothing ever checks whether a sentence is **true**. A big GPT writes much better text than ours, but it was trained on the same question we used, "what comes next?". So it can write things that sound right and are wrong. Our tiny GPT made up words like "rechess" because they look like the words around them. A big GPT can make up facts, quotes and references in the same way, because they look like the kind of text that usually comes next. This is really important to keep in mind any time you use these tools!

There are two more small differences you will see in the reading and in real code:

1. Real GPTs put the layer norm **before** attention and before the feed-forward part, instead of after. This trains more smoothly when there are lots of blocks.
2. They use dropout to help with overfitting.

Neither one changes the main idea.

**How does this connect to the reading?** Geron builds the **original** transformer design from the paper "Attention Is All You Need" (Vaswani et al. 2017) and uses it to translate English to Spanish. It has two halves:

- The **encoder** reads the English sentence with no mask. It can see the whole sentence at once, like our review classifier.
- The **decoder** writes the Spanish one word at a time with a causal mask, exactly like our tiny GPT. It also looks at what the encoder made, so it knows which English sentence it is translating.

A GPT is just the decoder half on its own, with no English sentence to look at. We will come back to the different kinds of transformers soon.

:::{.callout-note icon=false}
## Q6. Write this one out by hand

**a.** Name two things that are the SAME in our tiny GPT and GPT-3, and two things that are different.

**b.** The original transformer has an encoder and a decoder. Which half uses a causal mask, and why does that half need it?
:::

---

# Putting It Together

Here is everything we did today, in one list:

1. A GPT guesses the **next** piece of text. The targets are the input moved over by one, so the text is its own label.
2. The **context window** is how many pieces the model can see at once (64 characters for us).
3. The **causal mask** hides the future: every spot can only look at itself and the spots before it. In PyTorch it is `torch.triu(..., diagonal=1)`, with True where looking is NOT allowed.
4. A tiny GPT is embeddings plus positions, a stack of transformer blocks with the mask, and an `nn.Linear` that scores every possible next character at every spot.
5. To write, take the scores at the last spot, softmax, pick one at random, add it to the end, repeat. **Temperature** controls how often the model picks the less likely characters.
6. Without the mask the loss looks amazing and the model can not write at all.
7. Nothing in the training checks whether the text is true, only whether it is a likely next piece.

# New Commands Today

| Command | What it does | The thing that trips people up |
|---|---|---|
| `sorted(set(text))` | every different character, in order | without `sorted` the order (and the ids) can change between runs |
| `pieces.reshape(n_pieces, 65)` then `[:, :-1]` and `[:, 1:]` | input and targets, moved over by one | each piece needs 65 characters to make 64 questions |
| `torch.triu(torch.ones(n, n, dtype=torch.bool), diagonal=1)` | the causal mask | True means **hidden**, not allowed |
| `attention(X, X, X, attn_mask=mask)` | attention with the mask | the mask is (spots, spots), the same for every piece |
| `nn.ModuleList()` and `.append(...)` | a list of layers that PyTorch can train | a plain Python list hides the weights from the optimizer |
| `scores.reshape(-1, vocab_size)`, `y.reshape(-1)` | every spot becomes its own question for the loss | the `-1` means "work this number out for me" |
| `torch.softmax(scores / temperature, dim=0)` | probabilities for the next character | divide BEFORE the softmax |
| `torch.multinomial(probs, num_samples=1)` | picks at random using the probabilities | it gives back a tensor, use `.item()` to get the id |

:::{.callout-tip icon=false}
## You Try: optional code

Nothing here is collected. Work through it if you want the idea to stick.

**1.** Start the model with your own text, eg. `generate(model, "JULIET:", 300)` or `generate(model, "To be", 300)`. Remember every character in your start has to be in `chars`.

**2.** Train for 10 epochs instead of 5. Does the validation loss keep going down? Does the text look better?

**3.** Try `n_blocks=1` or `n_numbers=32`. How much does the validation loss go up when the model is smaller?
:::

# Before Next Class

1. In your lecture notes notebook, add your handwritten notes and answers to the questions.
2. Do the practice problems in `HW_day18.ipynb`. They are practice, not graded, and the solutions are in the same folder.
3. Check the class website for what is due this week.
4. Coming up next: tokens instead of characters, the difference between pretraining and fine-tuning, and what happens when you make these models bigger.

# Answers to the Q Boxes

Try every one of these by hand first! These are short summaries. Your own written answers should explain more, because writing it out yourself is how you learn it.

:::{.callout-note collapse="true"}
## Q1. The text is its own label

**a.** Input "to be or" and target "o be or" plus whatever character comes after "or" in the text (8 targets). Written as pairs: after "t" comes "o", after "to" comes " ", after "to " comes "b", and so on. One piece of 8 characters asks 8 questions.

**b.** The target is just the next character of the text, and the text already has it. Any text at all can be training data without a person deciding anything.
:::

:::{.callout-note collapse="true"}
## Q2. The mask

**a.** Without the mask every spot can look at the next character, which is exactly the answer it is asked for. It learns to copy it and gets a tiny loss. When it writes new text there is no next character to copy, so it never learned how to guess.

**b.** "s" can look at "F", "i", "r" and "s" (itself and everything before it). For 64 characters the True values are the spots above the diagonal: $(64 \times 64 - 64) / 2 = 2{,}016$.
:::

:::{.callout-note collapse="true"}
## Q3. No mean, and the loss to beat

**a.** The review classifier needed ONE answer per review. The GPT needs a guess for the next character at EVERY spot, so it keeps all 64 lists of numbers and scores each one.

**b.** $\ln(27) \approx 3.296$. A loss of 0 would mean the model gives probability 1 to the right next character every single time, which no model can do on real text.
:::

:::{.callout-note collapse="true"}
## Q4. Writing and temperature

**a.** It guesses one character, adds it to the end of the text, then sends the longer text back in and guesses the next one. It does this 400 times. Each time it only looks at the last 64 characters.

**b.** We divide the scores by the temperature before the softmax. 0.5 makes the differences between scores bigger, so the likely characters get even more of the probability and the text is safer. 1.5 makes the differences smaller, so unlikely characters get picked more. For something creative you would go above 1, but not so high that the text breaks.
:::

:::{.callout-note collapse="true"}
## Q5. Forgetting the mask

**a.** The model could see the answer. Every spot looked at the next character and copied it, so the loss measures how well it copies, not how well it guesses.

**b.** When we write, the last spot is the one we ask about, and there is no character after it yet. During training 63 of its 64 spots had the answer right next to them, so it mostly learned to copy, not to guess from what came before.
:::

:::{.callout-note collapse="true"}
## Q6. The real thing

**a.** The same: embeddings plus positions, transformer blocks with a causal mask, multi-head attention, a guess for the next piece at every spot, writing one piece at a time. Different: the size (number of blocks, heads, numbers per piece), tokens instead of characters, a much longer context window, and a LOT more training text.

**b.** The decoder, because it writes the Spanish one word at a time. While training, each Spanish word should only see the Spanish words before it, or it could copy the answer, just like our tiny GPT. The encoder reads the English all at once and does not need a mask.
:::